In [1]:
# Load all artifacts needed for hyperparameter optimisation.
# setup_mlflow() must be called before any mlflow.start_run().

import sys
sys.path.insert(0, '..')

import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

import time
import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import make_scorer

from src.estimators      import XGBoostDst, LightGBMDst
from src.mlflow_tracking import setup_mlflow
from src.runner   import fit_model, run_segment
from src.evaluate import compute_metrics, neg_storm_rmse
from src.splits import BOUNDARIES, PURGE_H, build_masks
from src.features import build_feature_sets, NEUTRON_ALL
from src.config import STORM_THR, K_HORIZONS, H_STAR, EXPERIMENT_NAME

from sklearn.model_selection import RandomizedSearchCV

# Reference Model Performance & Hyperparameter Optimisation

This notebook performs reference model evaluation and hyperparameter optimisation for XGBoost and LightGBM at $h^* = 7$h with MODEL_C feature set (OMNI + $\delta n$, 21 features). Results are saved to `models/hp_opt_results.pkl` for use in the main ML notebook.

In [2]:
feat_data  = pd.read_parquet('../data/processed/feat_split.parquet')
ctx_data   = joblib.load('../models/context_constants.pkl')
fs_data    = joblib.load('../models/feature_selection_results.pkl')

mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment('cosmic_ray_storm_prediction')

<Experiment: artifact_location='file:///C:/Temp/python-start/work/PracticalProjects/cosmic-ray-storm/notebooks/../mlruns/292621262165687921', creation_time=1782395978156, effective_trace_archival_retention=None, experiment_id='292621262165687921', last_update_time=1782395978156, lifecycle_stage='active', name='cosmic_ray_storm_prediction', tags={}, trace_location=None, workspace='default'>

## Reference Model Performance

Both XGBoost and LightGBM are first evaluated with the reference hyperparameter configuration on Train_1 + Train_2. This establishes a pre-tuning baseline against which the optimised models are compared. Since the forecasting horizon, feature set and training data remain fixed throughout, any performance differences after hyperparameter optimisation can be attributed to the optimisation process itself.

### Feature Sets

The predictor configuration is fixed from the scientific validation — MODEL_C (OMNI + $\delta n$, 21 features). Feature sets are constructed from the selected features using `src/features.py`.

In [3]:
feature_sets          = build_feature_sets(fs_data['selected_final'])
MODEL_C_OMNI_DNEUTRON = feature_sets['MODEL_C_OMNI_DNEUTRON']
OMNI_FEATURES         = feature_sets['OMNI_FEATURES']

print(f'MODEL_C features: {len(MODEL_C_OMNI_DNEUTRON)}')
print(MODEL_C_OMNI_DNEUTRON)

MODEL_C features: 21
['bz_gsm', 'sw_speed', 'sw_density', 'sw_pressure', 'e_field', 'mach_alfven', 'f107', 'ssn', 'bz_acc_3h', 'bz_acc_6h', 'bz_acc_12h', 'bz_gsm_lag1', 'bz_gsm_lag3', 'bz_gsm_lag12', 'bz_gsm_lag21', 'sw_speed_lag1', 'sw_speed_lag3', 'sw_speed_lag7', 'solar_sin', 'solar_cos', 'd_neutron']


### Data Splits

| Segment | Period | Role |
|---|---|---|
| Train_1 | 1995–2003-10-14 | Training |
| Val_Storm | 2003-10-15 – 2003-12-15 | External validation — extreme storm (excluded from training) |
| Train_2 | 2003-12-16 – 2008-12-31 | Training |
| Val_Main | 2009–2014 | External validation — routine conditions |

Train_1 + Train_2 combined (`train_mask`) is used for training and hyperparameter search. Val_Storm is excluded from `train_mask` despite falling within the 1995–2008 date range. Val_Main and Val_Storm are used only for post-optimisation evaluation — never during training or the hyperparameter search itself.

In [4]:
# ── Splits & Training data ────────────────────────────────────────────────
# build_masks() constructs all segment masks from src/splits.py
# train1_mask is needed only for the MASE denominator
masks          = build_masks(feat_data['datetime'])
train_mask     = masks['train']        # Train_1 + Train_2
train1_mask    =  masks['train1']
val_main_mask  = masks['val_main']
val_storm_mask = masks['val_storm']

EVAL_SEGMENTS = {
    'val_main' : val_main_mask,
    'val_storm': val_storm_mask,
}

X_train_full = feat_data.loc[train_mask, MODEL_C_OMNI_DNEUTRON]
y_train_full = feat_data.loc[train_mask, 'dst_target_7h']
y_train      = feat_data.loc[train1_mask, 'dst'].copy()  # MASE denominator

print(f'Train_1+2 rows     : {train_mask.sum():,}')
print(f'Val_main rows      : {val_main_mask.sum():,}')
print(f'Val_storm rows     : {val_storm_mask.sum():,}')
print(f'NaN in y_train_full: {y_train_full.isna().sum()}')

Train_1+2 rows     : 121,185
Val_main rows      : 52,542
Val_storm rows     : 1,446
NaN in y_train_full: 0


### Candidate Model Training

Both models are trained on Train_1 + Train_2 (1995–2008) with the reference hyperparameter configuration below. These results serve as the pre-tuning baseline.

| Parameter | XGBoost | LightGBM |
|---|---|---|
| `n_estimators` | 500 | 500 |
| `max_depth` | 5 | 5 |
| `learning_rate` | 0.05 | 0.05 |
| `subsample` | 0.8 | 0.8 |
| `colsample_bytree` | 0.8 | 0.8 |
| `num_leaves` | — | 31 |

In [5]:
# ── Candidate Model Training ──────────────────────────────────────────────
# Reference hyperparameters — pre-tuning baseline
xgb_pipe  = Pipeline([('model', XGBoostDst())])
xgb_pipe.fit(X_train_full, y_train_full)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['bz_gsm', 'sw...ed', 'sw...ty', 'sw...re', ...]"
,n_estimators,500
,max_depth,5
,learning_rate,0.05
,subsample,0.8
,colsample_bytree,0.8


In [6]:
lgbm_pipe = Pipeline([('model', LightGBMDst())])
lgbm_pipe.fit(X_train_full, y_train_full)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['bz_gsm', 'sw...ed', 'sw...ty', 'sw...re', ...]"
,n_estimators,500
,max_depth,5
,learning_rate,0.05
,num_leaves,31
,subsample,0.8


### Reference Performance — Results

Both reference models are evaluated on Val_Main and Val_Storm before hyperparameter optimisation. These results establish the pre-tuning baseline against which the optimised models will be compared.

In [7]:
# ── Reference Performance ─────────────────────────────────────────────────
print('\n── Reference performance (before tuning) ────────────────────────')
print(f'{"Model":<12} {"Segment":<12} {"RMSE":>8} {"StormRMSE":>12} {"MASE":>8}')
print('─' * 55)

for model_name, pipe in [('XGBoost', xgb_pipe), ('LightGBM', lgbm_pipe)]:
    for seg_name, seg_mask in EVAL_SEGMENTS.items():
        m = compute_metrics(
            y_true    = feat_data.loc[seg_mask, 'dst_target_7h'],
            y_pred    = pipe.predict(feat_data.loc[seg_mask, MODEL_C_OMNI_DNEUTRON]),
            y_train   = y_train,
            y_persist = feat_data.loc[seg_mask, 'dst'].values,
            storm_thr = STORM_THR,
            horizon   = H_STAR,
        )
        print(f'{model_name:<12} {seg_name:<12} '
              f'{m["rmse"]:>8.2f} {m["storm_rmse"]:>12.2f} {m["mase"]:>8.3f}')


── Reference performance (before tuning) ────────────────────────
Model        Segment          RMSE    StormRMSE     MASE
───────────────────────────────────────────────────────
XGBoost      val_main        14.48        27.29    3.435
XGBoost      val_storm       39.37       102.37    5.526
LightGBM     val_main        14.47        27.03    3.462
LightGBM     val_storm       40.01       104.25    5.608


## Hyperparameter Optimisation

The reference evaluation shows that both models exceed 100 nT Storm RMSE on Val_Storm — worse than the AR baseline (95.87 nT). Hyperparameter optimisation targets storm-period performance directly via `neg_storm_rmse` as the CV scorer.

**Search protocol:** RandomizedSearchCV (n_iter=50, random_state=42) with TimeSeriesSplit (5 folds) on Train_1 + Train_2. Val_Main and Val_Storm are not used during the search.

**Produces:** `../models/hp_opt_results.pkl` — best estimators, best params, CV results and reference metrics.

### Custom Scorer & TimeSeriesSplit

`neg_storm_rmse` returns negative Storm RMSE — RandomizedSearchCV maximises score, so higher = lower Storm RMSE = better model. `TimeSeriesSplit` ensures chronological order is preserved across folds. Val_Main and Val_Storm are never used during optimisation.

In [8]:
storm_scorer = make_scorer(neg_storm_rmse, greater_is_better=True)
tscv         = TimeSeriesSplit(n_splits=5)

### XGBoost RandomizedSearchCV

50 random combinations × 5 folds = 250 fits. Results logged to MLflow.

In [9]:
# ── XGBoost RandomizedSearchCV ────────────────────────────────────────────
#
# 50 random combinations × 5 folds = 250 fits
# Val_Main and Val_Storm are NOT used — scoring on CV folds only.

param_dist_xgb = {
    'model__n_estimators'    : [300, 500, 800, 1000],
    'model__max_depth'       : [3, 5, 7],
    'model__learning_rate'   : [0.005, 0.01, 0.05],
    'model__subsample'       : [0.7, 0.8, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 1.0],
}

xgb_search = RandomizedSearchCV(
    estimator           = Pipeline([('model', XGBoostDst())]),
    param_distributions = param_dist_xgb,
    n_iter              = 50,
    cv                  = tscv,
    scoring             = storm_scorer,
    n_jobs              = 1,
    verbose             = 1,
    random_state        = 42,
    refit               = True,
)

t0 = time.time()
xgb_search.fit(X_train_full, y_train_full)
print(f'\nXGBoost time     : {(time.time()-t0)/60:.1f} min')
print(f'XGBoost params   : {xgb_search.best_params_}')
print(f'XGBoost CV score : {xgb_search.best_score_:.4f}')

Fitting 5 folds for each of 50 candidates, totalling 250 fits

XGBoost time     : 38.1 min
XGBoost params   : {'model__subsample': 0.8, 'model__n_estimators': 500, 'model__max_depth': 5, 'model__learning_rate': 0.005, 'model__colsample_bytree': 0.8}
XGBoost CV score : -34.8043


In [10]:
mlflow.end_run()

with mlflow.start_run(run_name='xgb_randomizedsearch'):
    mlflow.set_tag('model_type',  'xgboost')
    mlflow.set_tag('search_type', 'RandomizedSearchCV')
    mlflow.set_tag('n_iter', 50)
    mlflow.log_params(xgb_search.best_params_)
    mlflow.log_metric('cv_storm_rmse', -xgb_search.best_score_)
    mlflow.sklearn.log_model(
        xgb_search.best_estimator_,
        name='best_pipeline',
        skops_trusted_types=[
            'src.estimators.xgboost_dst.XGBoostDst',
            'xgboost.core.Booster',
            'xgboost.sklearn.XGBRegressor',
        ]
    )
    print('MLflow run logged.')

MLflow run logged.


### LightGBM RandomizedSearchCV

Full grid has 972 combinations — exhaustive search would take several hours. `RandomizedSearchCV` with `n_iter=50` and `random_state=42` samples 50 random combinations × 5 folds = 250 fits. Results logged to MLflow.

In [11]:
# ── LightGBM RandomizedSearchCV ──────────────────────────────────────────
#
# Val_Main and Val_Storm are NOT used — scoring on CV folds only.

param_dist_lgbm = {
    'model__n_estimators'    : [300, 500, 800, 1000],
    'model__max_depth'       : [3, 5, 7],
    'model__learning_rate'   : [0.005, 0.01, 0.05],
    'model__subsample'       : [0.7, 0.8, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 1.0],
    'model__num_leaves'      : [15, 31, 63],
}

lgbm_search = RandomizedSearchCV(
    estimator           = Pipeline([('model', LightGBMDst())]),
    param_distributions = param_dist_lgbm,
    n_iter              = 50,
    cv                  = tscv,
    scoring             = storm_scorer,
    n_jobs              = 1,
    verbose             = 1,
    random_state        = 42,
    refit               = True,
)

t0 = time.time()
lgbm_search.fit(X_train_full, y_train_full)
lgbm_elapsed = (time.time() - t0) / 60

print(f'\nLightGBM RandomizedSearch time : {lgbm_elapsed:.1f} min')
print(f'LightGBM best params           : {lgbm_search.best_params_}')
print(f'LightGBM best CV score         : {lgbm_search.best_score_:.4f}')

Fitting 5 folds for each of 50 candidates, totalling 250 fits

LightGBM RandomizedSearch time : 26.5 min
LightGBM best params           : {'model__subsample': 1.0, 'model__num_leaves': 15, 'model__n_estimators': 500, 'model__max_depth': 3, 'model__learning_rate': 0.01, 'model__colsample_bytree': 1.0}
LightGBM best CV score         : -33.7432


In [12]:
# ── MLflow logging ────────────────────────────────────────────────────────
mlflow.end_run()

with mlflow.start_run(run_name='lgbm_randomizedsearch'):
    mlflow.set_tag('model_type',  'lightgbm')
    mlflow.set_tag('search_type', 'RandomizedSearchCV')
    mlflow.set_tag('n_iter', 50)
    mlflow.log_params(lgbm_search.best_params_)
    mlflow.log_metric('cv_storm_rmse', -lgbm_search.best_score_)
    mlflow.log_metric('search_time_min', lgbm_elapsed)
    mlflow.sklearn.log_model(
        lgbm_search.best_estimator_,
        name='best_pipeline',
        skops_trusted_types=[
            'src.estimators.lightgbm_dst.LightGBMDst',
            'lightgbm.basic.Booster',
            'lightgbm.sklearn.LGBMRegressor',
            'collections.OrderedDict',
        ]
    )
    print('MLflow run logged.')

MLflow run logged.


### Save Artifacts

Save best estimators and params for use in the main ML notebook. `hp_opt_results.pkl` contains full CV results for diagnostic plots.

In [20]:
ref_metrics = {}
for model_name, pipe in [('XGBoost', xgb_pipe), ('LightGBM', lgbm_pipe)]:
    ref_metrics[model_name] = {
        seg: eval_pipe(pipe, mask)
        for seg, mask in EVAL_SEGMENTS.items()
    }

xgb_ref_metrics  = ref_metrics['XGBoost']
lgbm_ref_metrics = ref_metrics['LightGBM']

# ── Save artifacts ────────────────────────────────────────────────────────
joblib.dump({
    'xgb_best_estimator' : xgb_search.best_estimator_,
    'xgb_best_params'    : xgb_search.best_params_,
    'xgb_cv_results'     : xgb_search.cv_results_,
    'lgbm_best_estimator': lgbm_search.best_estimator_,
    'lgbm_best_params'   : lgbm_search.best_params_,
    'lgbm_cv_results'    : lgbm_search.cv_results_,
    'xgb_ref_metrics'    : xgb_ref_metrics,
    'lgbm_ref_metrics'   : lgbm_ref_metrics,
}, '../models/hp_opt_resultstest.pkl')
print('Saved: ../models/hp_opt_results.pkl')

Saved: ../models/hp_opt_resultstest.pkl


In [25]:
print(ctx_data.keys())

dict_keys(['FEATURE_COLS', 'K_HORIZONS', 'STORM_THR', 'PURGE_H', 'BOUNDARIES'])
